# LangGraph 测试Notebook
---
目标：验证 LangGraph `StateGraph` 基础流转，建立 `GlobalState` 定义与各层节点骨架。

该Notebook是另外一个的改写，只保留真正实现了的逻辑。

In [2]:
# ── 依赖安装 ──────────────────────────────────────────────────────────────────
%pip install -q python-dotenv
%pip install -qU langchain-core langchain-ollama
%pip install -qU langgraph
%pip install -qU langchain-anthropic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:

# ── 导入与环境初始化 ───────────────────────────────────────────────────────────
import os,sys
# Jupyter notebooks don't define __file__, so locate the project root by
# searching upward for CLAUDE.md (the project marker file).
def _find_project_root(marker: str = "CLAUDE.md") -> str:
    path = os.path.abspath("")
    for _ in range(6):
        if os.path.exists(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise RuntimeError(f"Project root not found (searched for '{marker}')")

_root = _find_project_root()
if _root not in sys.path:
    sys.path.insert(0, _root)
print("Project root:", _root)
import getpass
import logging
from dotenv import load_dotenv

from langchain_ollama import ChatOllama
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_anthropic import ChatAnthropic

from langgraph.graph import StateGraph, START, END

from langchain.agents import create_agent
from pydantic import BaseModel, Field
from SkiLib.metatools.informative import get_tools as get_info_tools
from SkiLib.registry import SkillRegistry

# 从项目根目录 .env 加载 API Key 与追踪配置
load_dotenv()

# LangSmith 追踪（可选 — 在 .env 中设置 LANGSMITH_TRACING=true 启用）
if os.getenv("LANGSMITH_TRACING", "false").lower() == "true":
    if not os.getenv("LANGSMITH_API_KEY"):
        os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter LangSmith API Key: ")
    if not os.getenv("LANGSMITH_PROJECT"):
        os.environ["LANGSMITH_PROJECT"] = getpass.getpass("Enter LangSmith Project name: ")

logging.basicConfig(level=logging.ERROR, force=True)
print("Environment loaded. LangSmith tracing:", os.getenv("LANGSMITH_TRACING", "false"))

Project root: d:\code\skiagent\RoboSkiAgent
Environment loaded. LangSmith tracing: false


In [2]:
LLM_TYPE = "claude"  # 可选 "claude" 或 "ollama"（本地）
# ── LLM 配置 ──────────────────────────────────────────────────────────────────
# 使用本地 Ollama 时修改 OLLAMA_MODEL_ID 切换模型
OLLAMA_MODEL_ID = os.getenv("OLLAMA_MODEL_ID", "qwen3:latest")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

if LLM_TYPE == "claude":
    llm = ChatAnthropic(
        model="claude-sonnet-4-6"
    )
else:
    llm = ChatOllama(
        model=OLLAMA_MODEL_ID,
        base_url=OLLAMA_BASE_URL,
        temperature=0,
    )

# 快速连通性检查
try:
    _ping = llm.invoke("Reply with one word: ready")
    print("LLM reachable:", _ping.content.strip()[:80])
except Exception as e:
    print(f"[WARN] LLM not reachable — nodes using llm will fail. ({e})")

LLM reachable: ready


## GlobalState
与 `CLAUDE.md` 中 `GlobalState` 规范保持一致。  
`robot_state` 此处用 `dict` 代替，生产版本从 `SkiLib.base` 导入 `RobotState`。

In [ ]:
# ── GlobalState 定义 ──────────────────────────────────────────────────────────
from typing import TypedDict, Annotated, Optional, Literal
import operator


class GlobalState(TypedDict):
    # Layer-1：规划层输出
    todo_list: list[dict]           # [{task_id, type, skill/description, params}, ...]

    # Layer-2：执行上下文
    current_task: dict              # 执行槽：{} = 空闲，{...} = 执行中或失败保留

    # 机器人运行时快照（此处用 dict 代替，生产类型：SkiLib.base.RobotState）
    robot_state: dict

    # 控制标志
    halt_flag: bool                 # True = 所有 R-skill 执行被锁定
    halt_reason: Optional[str]      # "TASK_FAILURE" | "MANUAL_TASK" | None

    # Executor 写入；含 needs_hitl 字段供 Context Flush 决策
    last_result: Optional[dict]

    # 内部路由字段：HumanIntervention的结果, = "complete" | "abort"
    intervention_action: Optional[str]
    
    #解耦出的单独在HITL结束后的action， = "retry" | "next_task" | "replan" | "abort"
    hitl_command: Optional[str]

    # 审计日志，由 Context Flush 写入；Annotated list 避免键覆盖
    execution_log: Annotated[list[str], operator.add]

    # LangGraph 消息总线
    messages: Annotated[list[BaseMessage], operator.add]


print("GlobalState keys:", list(GlobalState.__annotations__.keys()))

GlobalState keys: ['todo_list', 'current_task', 'robot_state', 'halt_flag', 'halt_reason', 'last_result', '_hi_action', 'execution_log', 'messages']


In [4]:
# ── 节点：Supervisor（新版，create_agent, create_react_agent已经废弃）────────────────────────────────
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from SkiLib.metatools.informative import get_tools as get_info_tools
from SkiLib.registry import SkillRegistry


# ── SupervisorOutput Schema ───────────────────────────────────────────────────
# available_skills 不在此 schema 中 — 由代码从 SkillRegistry 注入，LLM 不填写。
class SupervisorOutput(BaseModel):
    """Fact sheet produced after knowledge saturation. Symbol-only, no coordinates."""
    task_intent_original: str = Field(description="Verbatim user instruction")
    
    task_intent: str = Field(
        description="Rewritten instruction using exact RoboDK symbol names"
    )
    
    scene: dict = Field(
        description="Keys: targets (list[str]), objects (list[str]), tools (list[str])"
    )
    
    # available_skills: This will be injected by code, not filled by LLM. Keys: skill name, docstring.
    
    extra_info: str = Field(
        default="",
        description="Unresolvable ambiguities or free-text observations"
    )


def _get_available_skills() -> dict:
    """Pure-code: read skill signatures from SkillRegistry. No LLM involved."""
    registry = SkillRegistry.instance()
    if not registry:
        return {}
    return {
        name: registry.get_skill(name).execute.__doc__ or ""
        for name in registry.list_skills()
    }


# ── System Prompt ─────────────────────────────────────────────────────────────
def _build_supervisor_prompt() -> str:
    skills_text = "\n".join(
        f"  - {name}: {doc.strip()}"
        for name, doc in _get_available_skills().items()
    ) or "  (none registered)"

    return f"""\
You are the Supervisor agent in an industrial robot assembly system.

Your ONLY job: gather scene facts and produce a structured SupervisorOutput summary.
Do NOT plan tasks, choose skills, or compute coordinates.

Rules:
- Call tools to query the RoboDK scene until you have enough information.
- Rewrite the instruction in `task_intent` using exact RoboDK symbol names 
and exact skill names (e.g. "Part_A_1", not "part A", "PickAndPlace" (A skill that exists) instead of "Pick and Place").
- All `scene` fields must use exact names returned by query tools — no invented names.
- Never include coordinates, joint angles, or numeric poses anywhere in your output.
- Record unresolvable ambiguities in `extra_info` instead of guessing.

Available skills (for your reference only — do not invent new ones):
{skills_text}
"""


# ── Lazy singleton ────────────────────────────────────────────────────────────
_supervisor_agent = None

def _get_supervisor_agent():
    global _supervisor_agent
    if _supervisor_agent is None:
        _supervisor_agent = create_agent(
            model=llm,
            tools=get_info_tools(),
            response_format=SupervisorOutput,
            system_prompt=_build_supervisor_prompt(),
        )
        print("[supervisor] Agent built. Skills:", list(_get_available_skills().keys()))
    return _supervisor_agent


# ── Node function ─────────────────────────────────────────────────────────────
def supervisor(state: GlobalState) -> dict:
    result = _get_supervisor_agent().invoke({"messages": state["messages"]})

    summary: SupervisorOutput | None = result.get("structured_response")
    if summary is None:
        #重新思考fallback逻辑：跳出？
        raw = state["messages"][-1].content
        raw_str = raw if isinstance(raw, str) else str(raw)
        summary = SupervisorOutput(
            task_intent_original=raw_str,
            task_intent=raw_str,
            scene={},
            extra_info="response_format unsupported — stub fallback",
        )


    output = summary.model_dump()
    # Inject available_skills from registry (pure code, not LLM output)
    output["available_skills"] = _get_available_skills()

    return {"messages": [AIMessage(content=str(output))]}


print("Supervisor cell loaded.")

Supervisor cell loaded.


In [6]:
# ── Supervisor 单元测试 ───────────────────────────────────────────────────────
# 前提：RoboDK 已启动
from SkiLib.robotcontext import RobotContext
from SkiLib.registry import SkillRegistry
import json
# 1. 拉起 RoboDK API
context = RobotContext()
RDK    = context.RDK
robot  = context.robot
skill_registry = SkillRegistry.instance()
print("[init] RobotContext OK, skills:", skill_registry.list_skills())

# 2. 重置 supervisor 单例（使其用已初始化的 SkillRegistry 重建）
_supervisor_agent = None

# 3. 构造最小初始状态
_test_state = {
    "messages":      [HumanMessage(content="把 Part_A_1 放到 Place Part A")],
    "todo_list":     [], "current_task": {}, "robot_state": {},
    "halt_flag":     False, "halt_reason": None,
    "last_result":   None, "_hi_action":  None, "execution_log": [],
}

# 4. 运行
print("\n[test] Running supervisor...")
_updates = supervisor(_test_state)

# 5. 解析输出
_out = _updates["messages"][-1].content

2026-03-25 12:55:24,416 [INFO] SkiLib.registry — SkillRegistry: registered 'DummySkill'
2026-03-25 12:55:24,419 [INFO] SkiLib.registry — SkillRegistry: registered 'PickAndPlace'


[PrimitiveRegistry] Registered: Grasp
[PrimitiveRegistry] Registered: Release
[PrimitiveRegistry] Registered: MoveJ
[PrimitiveRegistry] Registered: MoveL
[DummySkill] __init__ called. Received primitives: ['Grasp', 'Release', 'MoveJ', 'MoveL']
[init] RobotContext OK, skills: ['DummySkill', 'PickAndPlace']

[test] Running supervisor...
[supervisor] Agent built. Skills: ['DummySkill', 'PickAndPlace']


In [7]:
_out

'{\'task_intent_original\': \'把 Part_A_1 放到 Place Part A\', \'task_intent\': \'使用 PickAndPlace 技能，将工件 Part_A_1 从 Pick Part A（抓取点）经由 App Pick Part A（抓取接近点）拾取，再经由 App Place Part A（放置接近点）放置到 Place Part A（放置点）\', \'scene\': {\'targets\': [\'Pick Part A\', \'App Pick Part A\', \'Place Part A\', \'App Place Part A \'], \'objects\': [\'Part_A_1\'], \'tools\': [\'Gripper Extension\']}, \'extra_info\': \'当前夹爪 Gripper Extension 未抓取任何工件（grasped: []），可直接执行拾取操作。目标名称 "App Place Part A " 末尾含一个空格，已按场景中实际名称记录。\', \'available_skills\': {\'DummySkill\': \'Execute the dummy skill. Does nothing except log and return success.\', \'PickAndPlace\': \'\\n        Execute pick and place.\\n\\n        Args:\\n            item:            RoboDK name of the workpiece.\\n            pick_approach:   Approach/depart point near the pick location.\\n            pick_target:     Precise grasp point (MoveL).\\n            place_approach:  Approach/depart point near the place location.\\n            place_target:    Prec

In [8]:
#测试工具生成
skill_registry.get_tools()

[StructuredTool(name='DummySkill.check', description='Check that the dummy skill is reachable. message is echoed back in data.', args_schema=<class 'langchain_core.utils.pydantic.DummySkill.check'>, func=<function DummySkill.check at 0x0000025D8F6454E0>),
 StructuredTool(name='DummySkill.try_execute', description='Run check then execute. Aborts on check failure.', args_schema=<class 'langchain_core.utils.pydantic.DummySkill.try_execute'>, func=<function DummySkill.try_execute at 0x0000025D8F6FDDA0>),
 StructuredTool(name='PickAndPlace.check', description='Pre-flight feasibility check.\n\n        Args:\n            item:            RoboDK name of the workpiece to grasp/release.\n            pick_approach:   Target name for the approach/depart point near pick.\n            pick_target:     Target name for the linear-move precise grasp point.\n            place_approach:  Target name for the transit destination / depart point near place.\n            place_target:    Target name for the l

In [10]:
# ── 节点：Planner（工具调用版）────────────────────────────────────────────────
from pydantic import BaseModel, Field
from typing import Literal, Annotated, Union
from langchain_core.tools import StructuredTool
from langchain.agents import create_agent


# ── Task models（供下游 Dispatcher/Executor 读取）─────────────────────────────
class AutoTask(BaseModel):
    task_id: str
    type: Literal["auto"] = "auto"
    skill: str
    params: dict

class ManualTask(BaseModel):
    task_id: str
    type: Literal["manual"] = "manual"
    description: str

Task = Annotated[Union[AutoTask, ManualTask], Field(discriminator="type")]

class PlannerOutput(BaseModel):
    todo_list: list[Task]


# ── 动态工具生成 ───────────────────────────────────────────────────────────────
def _make_planner_tools(registry) -> tuple[list[StructuredTool], list[dict]]:
    """
    为每个 Skill 生成 add_<SkillName>_task 工具，直接复用 try_execute 的 args_schema。
    task_id 由代码自动追加（f"t{n}"），LLM 只需填写 skill 参数。
    """
    plan: list[dict] = []
    tools: list[StructuredTool] = []

    for skill_name in (registry.list_skills() if registry else []):
        skill = registry.get_skill(skill_name)
        try_exec = next(
            (t for t in skill.as_tools() if t.name.endswith(".try_execute")), None
        ) # only use try_execute for args_schema, ignore check
        if try_exec is None or try_exec.args_schema is None: # Use args_schema as kwargs later.
            continue

        
        # Generate a function shell for tool calling.
        def _create_task_adder(sname: str): # TODO: When decoupling this in future, pass plan to pass the reference
            def _add_task(**kwargs) -> str:
                task_id = f"t{len(plan) + 1}"
                plan.append({"task_id": task_id, "type": "auto", "skill": sname, "params": kwargs})
                return f"Task {task_id} ({sname}) added. Plan so far: {len(plan)} task(s)."
            return _add_task

        tools.append(StructuredTool(
            name=f"add_{skill_name}_task",
            description=f"Add a {skill_name} task. " + (try_exec.description or "").splitlines()[0],
            func=_create_task_adder(skill_name),
            args_schema=try_exec.args_schema,  # Reuse try_execute's args_schema for LLM input validation
        ))

    # Manual task 工具
    class AddManualTaskSchema(BaseModel):
        description: str = Field(description="What the human operator needs to do")

    def _add_manual(description: str) -> str:
        task_id = f"t{len(plan) + 1}"
        plan.append({"task_id": task_id, "type": "manual", "description": description})
        return f"Manual task {task_id} added."

    tools.append(StructuredTool.from_function(
        func=_add_manual,
        name="add_manual_task",
        description="Add a manual human-intervention step to the plan.",
        args_schema=AddManualTaskSchema,
    ))

    return tools, plan


_PLANNER_SYSTEM_PROMPT = """\
You are the planner of a symbolic robot assembly system.
Build a task plan by calling the provided tools — one call per task, in order.

Rules:
- Use ONLY exact symbol names from the scene in the user message — never invent names.
- Fill ALL required parameters (those without a default in the tool signature).
- Use add_manual_task when a step cannot be done by any provided tool.
- When done, stop calling tools. Do not output any JSON yourself.
"""


# ── Node function ─────────────────────────────────────────────────────────────
def planner(state: GlobalState) -> dict:
    print("[planner] Building plan via tool calls...")
    registry = SkillRegistry.instance()
    tools, plan = _make_planner_tools(registry)

    agent = create_agent(model=llm, tools=tools, system_prompt=_PLANNER_SYSTEM_PROMPT)

    # Supervisor 输出是 AIMessage — 重包成 HumanMessage，
    # 避免 Anthropic tool-use 模式下末尾 assistant 消息被拒。
    sup_content = state["messages"][-1].content
    agent.invoke({"messages": [HumanMessage(content=sup_content)]})

    manual_count = sum(1 for t in plan if t["type"] == "manual")
    print(f"[planner] Done: {len(plan)} tasks ({manual_count} manual)")

    return {
        "todo_list": plan,
        "messages": [AIMessage(content=f"[Planner] {len(plan)} tasks queued ({manual_count} manual)")],
    }


print("Planner cell loaded.")


Planner cell loaded.


In [12]:
# ── Planner 单元测试 ──────────────────────────────────────────────────────────
# 前提：RobotContext 已初始化（复用 supervisor 测试单元格中的 context）
# 直接从 RoboDK scene 查询真实符号名，不使用 mock

import json
from SkiLib.metatools.informative import list_targets, list_objects, list_tools

# 1. 从真实 scene 获取符号名
_real_targets = list_targets.invoke({})
_real_objects = list_objects.invoke({})
_real_tools   = list_tools.invoke({})
print("[scene] targets:", _real_targets)
print("[scene] objects:", _real_objects)
print("[scene] tools:  ", _real_tools)

# 2. 构造用户指令（使用 scene 里真实存在的第一个 object）
if not _real_objects:
    raise RuntimeError("No objects in RoboDK scene — cannot run Planner test.")
_test_object = "Part_A_1"
_user_instruction = f"把 {_test_object} 放到对应的放置点"

# 3. 构造 SupervisorOutput（task_intent 使用真实符号名，scene 来自真实查询）
#    _build_planner_prompt() 已经从 SkillRegistry 注入完整参数 schema，
#    这里不再需要 available_skills 字段。
_real_sup_output = json.dumps({
    "task_intent_original": _user_instruction,
    "task_intent": f"Pick {_test_object} and place it at the corresponding place target.",
    "scene": {
        "targets": _real_targets,
        "objects": _real_objects,
        "tools":   _real_tools,
    },
    "extra_info": "",
}, ensure_ascii=False, indent=2)

print("\n[SupervisorOutput passed to Planner]\n", _real_sup_output)

# 4. 构造 planner 入参 state
_planner_test_state = {
    "messages": [
        HumanMessage(content=_user_instruction),
        AIMessage(content=_real_sup_output),   # planner 读取 messages[-1]
    ],
    "todo_list": [], "current_task": {}, "robot_state": {},
    "halt_flag": False, "halt_reason": None,
    "last_result": None, "_hi_action": None, "execution_log": [],
}

# 5. 运行
print("\n[test] Running planner...")
_plan_updates = planner(_planner_test_state)

# 6. 打印结果
print("\n[PlannerOutput]")
for t in _plan_updates["todo_list"]:
    if t["type"] == "auto":
        print(f"  [{t['task_id']}] AUTO  skill={t['skill']}  params={t['params']}")
    else:
        print(f"  [{t['task_id']}] MANUAL  desc={t['description']!r}")

# 7. 验证：auto task 的 params 必须非空
auto_tasks = [t for t in _plan_updates["todo_list"] if t["type"] == "auto"]
empty_params = [t for t in auto_tasks if not t["params"]]
if empty_params:
    print(f"\n[FAIL] {len(empty_params)} auto task(s) have empty params: {[t['task_id'] for t in empty_params]}")
else:
    print(f"\n[PASS] All {len(auto_tasks)} auto task(s) have non-empty params.")

# 8. Pydantic 二次验证
_validated = PlannerOutput(todo_list=_plan_updates["todo_list"])
print(f"[PASS] Pydantic validation OK ({len(_validated.todo_list)} tasks)")


[scene] targets: ['Pick Part A New', 'Home A', 'Place Part A', 'App Place Part A ', 'App Pick Part A', 'Pick Part A', 'Pick Part B', 'App Pick Part B', 'App Place Part B', 'Place Part B', 'Home B', 'Pick Part C', 'App Pick Part C', 'Place Part C', 'App Place Part C', 'Home C', 'Target 6']
[scene] objects: ['Base Cylinder', 'New Table Actual', 'New Table Actual-22', 'New Table Actual-21', 'New Table Actual-20', 'New Table Actual-43', 'New Table Actual-44', 'New Table Actual-45', 'New Table Actual-40', 'New Table Actual-52', 'New Table Actual-53', 'Part_A_1', 'Part_A_2', 'Part_B_1', 'Part_B_2', 'Part_C_1', 'Part_C_2']
[scene] tools:   ['Gripper Extension']

[SupervisorOutput passed to Planner]
 {
  "task_intent_original": "把 Part_A_1 放到对应的放置点",
  "task_intent": "Pick Part_A_1 and place it at the corresponding place target.",
  "scene": {
    "targets": [
      "Pick Part A New",
      "Home A",
      "Place Part A",
      "App Place Part A ",
      "App Pick Part A",
      "Pick Part A",

In [ ]:
# Dispatcher 节点实现
# TODO: 检查Halt逻辑,是否该在这里实现?
def dispatcher(state: GlobalState) -> dict:
    # Slot occupied — do not overwrite (failed task kept for retry)
    if state["current_task"]:
        print(f"[dispatcher] Slot occupied: {state['current_task']['task_id']}, skip pop")
        return {}
    if not state['todo_list']:
        print("[dispatcher] No tasks in todo_list")
        return {}
    
    next_task = state['todo_list'][0]
    print(f"[dispatcher] Dispatching task {next_task['task_id']} ({next_task['type']}): {next_task['skill']}")
    task_remaining = state['todo_list'][1:]  #avoid mutating
    
    return {
        "current_task": next_task,
        "todo_list": task_remaining
    }
    
def task_router(state: GlobalState) -> str:
    if not state['current_task']:
        return "END"
    
    return state['current_task']['type']  # "auto" or "manual"



In [ ]:
# Manual Intervention Handler 实现 TODO: 目前是Stub
def manual_intervention_handler(state: GlobalState) -> dict:
    task = state.get("current_task")
    print(f"[manual_handler] Handling manual task: {task.get('description')}")
    
    
    # ========= 人工干预处理（此处为 Stub）=========
    
    
    # TODO:实际调用HITL
    return {
        "execution_log": [f"[manual_handler] Manual task handled -> SUCCESS (stub)"],
        "intervention_action": "complete",  
    }

def manual_intervention_router(state: GlobalState) -> str:
    action = state.get("intervention_action")
    if action == "complete":
        return "dispatcher"  # 回到 Dispatcher 分配下一个任务
    else: # 包括 "abort" 和未设置的情况
        if not action == "abort":
            print("[manual_intervention_router] Warning: unexpected intervention_action, aborted:", action) 
        return "END"
    

In [ ]:
# Executor & PostTaskHandler 实现 TODO: 目前是Stub
from SkiLib.base import ERROR_ROBOT_INACTIVE


def executor(state: GlobalState) -> dict:
    task = state.get("current_task")
    
    if state.get("halt_flag"):
        return {
            "execution_log": [f"[executor] 已挂起 — 跳过 {task.get('task_id')}"],
            "last_result": {"success": False, "error_type": ERROR_ROBOT_INACTIVE, "needs_hitl": True},
        }
    
    
    print(f"[executor] Running: {task.get('skill')}({task.get('params')})")
    
    # ========= LLM 调用执行器（此处为 Stub）=========
    
    
    # TODO：实际调用LLM
    
    
    return {
        "messages": [AIMessage(content=f"[Executor] Successfully executed {task.get('task_id')} ({task.get('skill')})")],
        "execution_log": [f"[executor] {task['task_id']} {task.get('skill')} -> SUCCESS (stub)"],
        "last_result": {"success": True, "error_type": None, "needs_hitl": False},
    }

def post_task_router(state: GlobalState) -> str:
    last_result = state.get("last_result")
    if last_result is None:
        print("[post_task_router] Warning: last_result is None, routing to END by default.")
        return "END"
    if last_result['success']:
        return "dispatcher"  # 继续分配下一个任务
    else:
        return "hitl_handler"





In [ ]:
# Human-in-the-loop Handler 实现 TODO: 目前是Stub
def hitl_handler(state: GlobalState) -> dict:
    # TODO:是否需要设置halt锁？
    # TODO : 实际调用 HITL
    
    command = "next_task"  # Stub: 默认选择继续下一个任务
    #command = Interrupt().... command = retry | next_task | replan | abort
    result = {
        "hitl_command": command
    }
    if command == "replan":
        # TODO: 把Prompt解耦出去统一配置，然后可能还需要修改
        result["messages"] = [HumanMessage(content="[HITL] As a skill failed to execute, you are requested to" + # type: ignore
            " replan the whole sequence. Please review the original instruction and the failed task, \
                then call the provided tools to build a new plan.")] # type: ignore
    
    if command == "next_task":
        result['current_task'] = {}  # 清空 current_task，允许 Dispatcher 分配下一个任务 #type: ignore
    return result

def hitl_router(state: GlobalState) -> str:
    command = state.get("hitl_command")
    if command == "retry":
        return "executor"
    elif command == "replan":
        return "supervisor"
    elif command == "next_task":
        return "dispatcher"
    else:  # 包括 "abort" 和未设置的情况
        if command not in ("abort", None):
            print("[hitl_router] Warning: unexpected hitl_command, routing to END:", command)
        return "END"

In [ ]:
# 图构建与可视化验证
